# LanceDB Fundemantals

In [1]:
import lancedb 
from constants import KNOWLEDGE_BASE_PATH

vector_db = lancedb.connect(KNOWLEDGE_BASE_PATH)
vector_db

LanceDBConnection(uri='/Users/omeraytug/Desktop/llmops/code_alongs/09_lancedb/knowledge_base')

In [2]:
import json

with open("animals_text_embeddings.json") as file:
    data = json.load(file)

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [3]:
vector_db.create_table("animals_text", exist_ok=True, data=data)
vector_db["animals_text"]

LanceTable(name='animals_text', version=1, _conn=LanceDBConnection(uri='/Users/omeraytug/Desktop/llmops/code_alongs/09_lancedb/knowledge_base'))

In [4]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"


## Add more data

In [12]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]

vector_db["animals_text"].add(more_data)

AddResult(version=2)

In [13]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Create empty table

In [ ]:
from lancedb.pydantic import LanceModel

class Article(LanceModel):
    title: str
    content:str


vector_db.create_table("articles", schema=Article, exist_ok=True)

/var/folders/qh/2zypz2ld4qsd044652nngpsc0000gn/T/ipykernel_19800/2422514638.py:10: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  vector_db.table_names()


['animals_text', 'articles']

In [20]:
vector_db.list_tables()

ListTablesResponse(tables=['animals_text', 'articles'], page_token=None)

TODO: Add data to articles

In [ ]:
from faker import Faker # artifical data generated by the package "faker"

fake = Faker()

articles_data = []

for x in range(5):
    articles_data.append({
        "title": fake.sentence(nb_words=3),
        "content": fake.paragraph(nb_sentences=3)
    })

articles_data

[{'title': 'Image a image.',
  'content': 'Some serious store reason someone thing account. Personal around right song outside.'},
 {'title': 'I adult treatment.',
  'content': 'Tough culture south prove. Man manager window couple large charge yeah.'},
 {'title': 'Environment eat.',
  'content': 'Point none soon leader majority degree ability these. Fact off child cause coach war.'},
 {'title': 'Sell method market.',
  'content': 'Open another figure more person on. Provide anyone fire religious accept great.'},
 {'title': 'Last same go.',
  'content': 'Prepare suddenly account by. Region change old born. Ago thus war.'}]

In [22]:
vector_db["articles"].add(articles_data)

AddResult(version=2)

In [23]:
vector_db["articles"].to_pandas()

,title,content
0,Image a image.,Some serious store reason someone thing accoun...
1,I adult treatment.,Tough culture south prove. Man manager window ...
2,Environment eat.,Point none soon leader majority degree ability...
3,Sell method market.,Open another figure more person on. Provide an...
4,Last same go.,Prepare suddenly account by. Region change old...


## Drop a table

In [24]:
vector_db.drop_table("articles")

In [25]:
vector_db.list_tables()

ListTablesResponse(tables=['animals_text'], page_token=None)

## Vector search in LanceDB

Flow here (common approach in many vector databases):
1. use embedding model to create vector embeddings for each document
2. use same embedding model to create a vector embedding of the query
3. send in the query_vector to search for the closest document

If we want to do RAG -> send in the closes document to LLM to use as context

In [37]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [38]:
query_vector = [0.5, 0.2, 0.9]

vector_db["animals_text"].search(query_vector).limit(3).to_pandas()

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]",0.2413
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]",0.2673


## Embeddings API in LanceDB

- makes life simpler
- automatically generate embeddings

cohere embeddings models:
https://docs.cohere.com/docs/cohere-embed

In [46]:
from lancedb.embeddings import get_registry
from dotenv import load_dotenv

load_dotenv()

embedding_model = get_registry().get("cohere").create(name = "embed-multilingual-v3.0")

embedding_model.ndims()

1024

In [47]:
from lancedb.pydantic import Vector

class JokeModel(LanceModel):
    joke: str = embedding_model.SourceField()
    embedding: Vector(embedding_model.ndims()) = embedding_model.VectorField() # type:ignore

vector_db.create_table("jokes", schema=JokeModel, exist_ok=True)
vector_db["jokes"]

LanceTable(name='jokes', version=1, _conn=LanceDBConnection(uri='/Users/omeraytug/Desktop/llmops/code_alongs/09_lancedb/knowledge_base'))

In [52]:
import pandas as pd
with open("jokes.json", encoding="utf8") as file:
    jokes_data = json.load(file)

df_jokes = pd.DataFrame(jokes_data)
df_jokes.head(5)

,joke
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


In [53]:
vector_db["jokes"].add(df_jokes)

AddResult(version=2)

In [56]:
df_jokes_embeddings = vector_db["jokes"].to_pandas()

df_jokes_embeddings.head(5)

,joke,embedding
0,Parallel lines have so much in common—it’s sad...,"[-0.0032196045, 0.0011167526, -0.014953613, 0...."
1,"ETL stands for “Extract, Transform, Leave for ...","[0.012634277, 0.005268097, -0.09136963, 0.0809..."
2,What do you call a snake that runs your script...,"[0.019165039, 0.044677734, -0.021530151, 0.028..."
3,"Gold walks into a bar. The bartender says, “Au...","[0.019866943, 0.05999756, 0.012702942, 0.02343..."
4,C# devs don’t argue; they just throw exceptions.,"[0.013122559, 0.002374649, -0.05706787, 0.0608..."


In [ ]:
df_jokes_embeddings["embedding"][0] # note dimension 1024

array([-0.0032196 ,  0.00111675, -0.01495361, ...,  0.06707764,
        0.00904846,  0.00076342], shape=(1024,), dtype=float32)

In [61]:
vector_db["jokes"].search("chemistry related joke").to_pandas()

,joke,embedding,_distance
0,I told a chemistry joke… there was no reaction.,"[-0.03274536, 0.018218994, 0.039276123, 0.0012...",0.810664
1,Chemists have all the solutions… mostly in bea...,"[0.0013589859, 0.03086853, -0.0012874603, 0.02...",1.003894
2,Why did the chemist ground his kids? Because t...,"[-0.008056641, 0.031555176, -0.019454956, 0.01...",1.016106
3,Oxygen and Magnesium got together—OMg!,"[-0.04119873, -0.004600525, 0.007701874, 0.006...",1.024265
4,The C# compiler walked into a bar. The bartend...,"[0.0038490295, 0.006790161, -0.0769043, 0.0265...",1.115635
5,Never trust an atom—they make up everything.,"[-0.008956909, 0.003440857, -0.020614624, 0.04...",1.124924
6,Why’s 6 afraid of 7? Because 7 8 9.,"[0.005886078, 0.020553589, -0.0031318665, 0.01...",1.137697
7,"Gold walks into a bar. The bartender says, “Au...","[0.019866943, 0.05999756, 0.012702942, 0.02343...",1.168893
8,I put “root beer” in a square glass… now it’s ...,"[0.008476257, 0.026138306, 0.046020508, 0.0015...",1.172221
9,My math teacher said I’m average… how mean!,"[-0.0026016235, 0.03112793, 0.013343811, 0.030...",1.244877


In [ ]:
# Since the embedding model is multilingual, we could also search in other languages despite the data being English.

vector_db["jokes"].search("kimya").to_pandas()

,joke,embedding,_distance
0,Oxygen and Magnesium got together—OMg!,"[-0.04119873, -0.004600525, 0.007701874, 0.006...",1.209664
1,I told a chemistry joke… there was no reaction.,"[-0.03274536, 0.018218994, 0.039276123, 0.0012...",1.212255
2,Chemists have all the solutions… mostly in bea...,"[0.0013589859, 0.03086853, -0.0012874603, 0.02...",1.234572
3,Never trust an atom—they make up everything.,"[-0.008956909, 0.003440857, -0.020614624, 0.04...",1.243880
4,I tried to explain async/await to my friend… n...,"[0.0028648376, 0.007926941, -0.016326904, 0.00...",1.321968
5,I put “root beer” in a square glass… now it’s ...,"[0.008476257, 0.026138306, 0.046020508, 0.0015...",1.360302
6,My Python code works on the first try… said no...,"[0.02017212, 0.035736084, -0.008163452, 0.0485...",1.379902
7,The C# compiler walked into a bar. The bartend...,"[0.0038490295, 0.006790161, -0.0769043, 0.0265...",1.380364
8,Python is so friendly even whitespace gets a say.,"[0.0146484375, 0.016906738, -0.0026435852, 0.0...",1.395953
9,Why did the chemist ground his kids? Because t...,"[-0.008056641, 0.031555176, -0.019454956, 0.01...",1.396658
